# ЛР 1. Начало работы с ML-проектом: формализация задачи и первичный анализ данных
<!-- В VSCode есть специальное расширение для генерации интерактивного оглавления -->
<!-- Желательно в оглавлении сохранить нумерацию пунктов -->
<!-- Правилом хорошего тона является создание интерактивных заголовков пунктов и подпунктов, которые ведут на оглавление -->

**Table of contents**<a id='toc0_'></a>

## Описание задачи
**1. Предметная область и боли заказчика:**
Заказчиком выступает король Эйгон I Таргариен, который строит Красный замок. Крепость должна выдерживать осады и быть максимально надежной. Мейстеры провели испытания строительной смеси. 

*Боль заказчика (Эйгона):* Он не хочет слепо доверять мастерам и желает знать наверняка: от чего сильнее зависит прочность стены — от самой смеси или от времени её выдержки? Строительство нельзя тормозить просто так, поэтому нужно обосновать, имеет ли смысл долго ждать усадки.

*Суть гипотезы Эйгона:* Влияет ли время на итоговую прочность стен так же сильно (или даже сильнее), чем состав смеси?

**2. Постановка ML-задачи:**
Перед нами стоит задача **регрессии** (Regression), так как целевая переменная (прочность на сжатие) является непрерывной вещественной величиной. Дополнительной (и главной) задачей является **анализ важности признаков (гипотеза Эйгона)**, чтобы ответить на вопрос короля о влиянии состава смеси против влияния времени.

**3. Объект задачи:**
Объектом исследования является отдельный образец строительной смеси, для которого известен состав, время выдержки и итоговая прочность.

**4. Образ выходного результата модели:**
На выходе модель должна предсказывать точное число — прочность материала на сжатие. Также результатом работы станет аналитический вывод о том, какой фактор вносит наибольший вклад в прочность.

`ячейки для описания исходных данных`
<!-- 
    В этой и последующих ячейках предлагается описать исходные данные, а также сопутствующую информацию:
    1. таблицу с семантикой данных, а также типами данных столбцов (ожидаемый тип, а не фактический, которые мы получаем при чтении данных) в формате MD (кодом не надо выводить таблички)
    2. формулы прикладной области (с указанием, какие формулы можно использовать, а какие -- нет)
-->

## Описание исходных данных и их семантика (Data Understanding)

Набор данных представляет собой результаты испытаний образцов строительной смеси. Все признаки являются числовыми (вещественными).

| Название признака (столбец) | Семантика (описание) | Ожидаемый тип данных | Роль |
| :--- | :--- | :--- | :--- |
| **Cement** | Количество цемента в смеси (кг/м³) | `float` | Входной |
| **Blast Furnace Slag** | Доменной шлак (кг/м³) | `float` | Входной |
| **Fly Ash** | Зола-унос (кг/м³) | `float` | Входной |
| **Water** | Количество воды (кг/м³) | `float` | Входной |
| **Superplasticizer** | Суперпластификатор (кг/м³) | `float` | Входной |
| **Coarse Aggregate** | Крупный заполнитель / щебень (кг/м³) | `float` | Входной |
| **Fine Aggregate** | Мелкий заполнитель / песок (кг/м³) | `float` | Входной |
| **Age** | Время выдержки материала (в днях) | `float` | Входной |
| **Concrete compressive strength** | Прочность на сжатие (МПа) | `float` | **Целевой** |

`Доменный шлак` — это побочный продукт, который образуется при выплавке чугуна в доменной печи. Проще говоря, это «отход» металлургического производства, который раньше просто выбрасывали, а сейчас активно используют в строительстве. 

`Зола-унос (или летучая зола)` — это тонкодисперсный порошок, который образуется при сжигании твёрдого топлива, чаще всего угля, на тепловых электростанциях.

`Суперпластификатор` — это такая «волшебная» добавка в бетонную смесь, которая делает её гораздо более пластичной и текучей, чем обычно. При этом не нужно добавлять лишнюю воду, чтобы смесь было легко заливать.

## Формулы прикладной области

### Водовяжущее отношение 
— это отношение массы воды к массе вяжущего вещества после затворения смеси. Этот параметр используется в строительстве, в частности при работе с бетонными смесями, где часть цемента может заменяться минеральными добавками (например, золой-уноса, доменным гранулированным шлаком, микрокремнезёмом). 

Новый признак (Binder): $Binder = Cement + Slag + Fly Ash$

Новый признак (W/B): $W/B = \frac{Water}{Binder}$

---
### Формула Боломея-Скрамтаева
Для бетонов средней и высокой прочности закон Абрамса часто заменяют линейной формулой Боломея-Скрамтаева, которая использует обратное отношение (цемент к воде):$$R = A \cdot R_c \cdot \left( \frac{C}{W} - 0.5 \right)$$Где $C/W$ — цементно-водное отношение.Новый признак: $\frac{Cement}{Water}$ или $\frac{Binder}{Water}$. 

Поскольку $A$ и $R_c$ для конкретной выборки являются константами (или скрытыми параметрами), задача алгоритма — самостоятельно подобрать такие веса, которые заменят эти неизвестные множители.

`справка` $R_c$ (Активность или марка цемента): Это физико-механическая характеристика самого цемента (например, цемент марки М400 или М500). $A$ (Коэффициент качества): Это эмпирический коэффициент, зависящий от качества заполнителей (щебня и песка). 

---
### Логарифмический закон роста прочности во времени 
— это эмпирическая зависимость, которая приближённо описывает процесс увеличения прочности материалов (например, бетона на основе цементного вяжущего) с течением времени. 

`Суть закона`

Согласно этому закону, прочность материала сначала растёт интенсивно, а затем темп замедляется. Математически это выражается тем, что зависимость прочности от времени в обычном масштабе является логарифмической функцией, а в логарифмическом масштабе времени — линейной.

`Формула расчёта`

$$R_n = R_{28} \cdot \frac{\log(n)}{\log(28)}$$

где:
Rₙ — прочность бетона в возрасте n суток;
R₂₈ — прочность в возрасте 28 суток;

Новый признак: $Log\_Age = \log(Age)$

---

В строительстве (и в записях мейстеров) базовым правилом оценки прочности является **Закон Абрамса (Abrams' law)**. Он гласит, что прочность смеси обратно пропорциональна водоцементному отношению (отношению массы воды к массе цемента):

$$ S = \frac{A}{B^{W/C}} $$

Где:
* $S$ — прочность материала
* $W/C$ — отношение массы воды к массе цемента (признак Water разделить на признак Cement)
* $A$ и $B$ — константы, зависящие от внешних условий.

### Вывод: использование формул в ML-задаче

Опираясь на описанные выше физические законы прочности, мы должны четко разделить их применимость в рамках нашего ML-проекта:

**Что использовать НЕЛЬЗЯ:**
Нельзя применять формулу Боломея-Скрамтаева или закон Абрамса напрямую для аналитического (ручного) расчета целевой переменной (прочности на сжатие), подменяя тем самым модель машинного обучения. Эти эмпирические формулы идеализированы, они требуют знания скрытых коэффициентов (качество заполнителей $A$, истинная марка цемента $R_c$) и не умеют корректно учитывать влияние современных химических добавок (например, суперпластификатора). Алгоритм должен сам найти веса и зависимости.

**Что использовать МОЖНО:**
Можно (и нужно) использовать математическую **структуру** этих формул на этапе генерации новых признаков (Feature Engineering). Мы имеем полное право создать новые столбцы в нашем датасете, опираясь на физику процесса:
1. Водовяжущее отношение ($W/B$) или обратное ему ($Binder/Water$).
2. Логарифм возраста ($Log\_Age$).

`ячейки для создания и фиксации виртуального окружения`
<!-- 
    В этой и последующих ячейках предлагается зафиксировать информацию о виртуальном окружении, а именно:
    1. версию питона
    2. версию и список используемых библиотек
    3. основные и часто встречающиеся импорты (pandas, numpy, typing, warnings и т.д.)
    4. случайный сид (seed / random_state)
-->

In [2]:
!python --version # Версия Python

Python 3.13.2


In [3]:
!pip list # Список и версии библиотек

Package                   Version
------------------------- ------------------
absl-py                   2.4.0
altair                    6.1.0
annotated-doc             0.0.4
anyio                     4.13.0
asttokens                 3.0.1
astunparse                1.6.3
attrs                     26.1.0
beautifulsoup4            4.15.0
blinker                   1.9.0
cachetools                7.1.4
catboost                  1.2.10
certifi                   2026.5.20
cffi                      2.0.0
charset-normalizer        3.4.7
click                     8.4.1
cloudpickle               3.1.2
colorama                  0.4.6
comm                      0.2.3
contourpy                 1.3.3
cycler                    0.12.1
Cython                    3.2.4
DAWG2-Python              0.9.0
debugpy                   1.8.20
decorator                 5.2.1
defusedxml                0.7.1
et_xmlfile                2.0.0
executing                 2.2.1
filelock                  3.29.1
flatbuffers   

In [4]:
# Базовые библиотеки для работы с таблицами и массивами
import pandas as pd
import numpy as np

# Настройка предупреждений
import warnings
warnings.filterwarnings('ignore')

# Инструмент для разделения выборки
from sklearn.model_selection import train_test_split

# Фиксация случайного сида для воспроизводимости
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# 1. Предварительная обработка данных

## 1.1. Чтение и загрузка данных 

In [5]:
# Загрузка данных
df = pd.read_csv('../../data/game_of_thrones.csv', sep=';')

In [6]:
df.head(20)

,Cement kg/m^3,Blast Furnace Slag kg/m^3,Fly Ash kg/m^3,Water kg/m^3,Superplasticizer kg/m^3,Coarse Aggregate kg/m^3,Fine Aggregate kg/m^3,Age day,Concrete compressive strength
0,540.0,0.0,0.0,NaN,2.5,1040.0,676.0,28.0,79.99
1,540.0,0.0,0.0,162.0,2.5,1055.0,676.0,28.0,61.89
2,332.5,142.5,0.0,228.0,0.0,932.0,594.0,270.0,40.27
3,332.5,142.5,0.0,228.0,0.0,932.0,594.0,365.0,41.05
4,198.6,132.4,0.0,192.0,0.0,978.4,825.5,360.0,44.30
5,266.0,114.0,0.0,228.0,0.0,932.0,NaN,90.0,47.03
6,380.0,95.0,0.0,228.0,0.0,932.0,594.0,365.0,43.70
7,380.0,95.0,0.0,228.0,0.0,932.0,594.0,28.0,36.45
8,266.0,114.0,0.0,228.0,0.0,932.0,670.0,28.0,45.85
9,475.0,0.0,0.0,228.0,0.0,932.0,594.0,28.0,39.29


## 1.2. Первичный анализ данных

Первоначально необходимо проанализировать типы данных колонок датафрейма и проверить наличие пропущенных значений. Воспользуемся методом датафрейма `.info()`, который выводит сводную информацию об объекте. Дополнительно применим метод `.describe()` для оценки распределения (среднее, минимум, максимум, квартили) числовых признаков. 
Это поможет нам понять, готовы ли данные к делению на обучающую и тестовую выборки.

In [7]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1030 entries, 0 to 1029
Data columns (total 9 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   Cement kg/m^3                  968 non-null    float64
 1   Blast Furnace Slag kg/m^3      958 non-null    float64
 2   Fly Ash kg/m^3                 948 non-null    float64
 3   Water kg/m^3                   948 non-null    float64
 4   Superplasticizer kg/m^3        937 non-null    float64
 5   Coarse Aggregate kg/m^3        927 non-null    float64
 6   Fine Aggregate kg/m^3          906 non-null    float64
 7   Age day                        978 non-null    float64
 8   Concrete compressive strength  1030 non-null   float64
dtypes: float64(9)
memory usage: 72.6 KB


1. **Типы данных:** Все 9 признаков корректно спарсились как вещественные числа (`float64`). Приведение типов не требуется, данные соответствуют заявленной семантике.

2. **Целевая переменная:** Признак `Concrete compressive strength` не имеет пропусков (1030 non-null из 1030). Это значит, что удаление строк из-за отсутствия целевой метки не потребуется.

3. **Наличие пропущенных значений:** Во всех входных признаках (от `Cement` до `Age`) присутствуют пропущенные значения (NaN), так как количество non-null значений везде меньше 1030. Следовательно, на этапе предварительной обработки будет необходимо применить стратегии заполнения пропусков. 

In [8]:
df.describe()

,Cement kg/m^3,Blast Furnace Slag kg/m^3,Fly Ash kg/m^3,Water kg/m^3,Superplasticizer kg/m^3,Coarse Aggregate kg/m^3,Fine Aggregate kg/m^3,Age day,Concrete compressive strength
count,968.000000,958.000000,948.00000,948.000000,937.000000,927.000000,906.000000,978.000000,1030.000000
mean,280.409814,73.453758,53.48692,181.708228,6.202455,972.645847,773.029470,45.721881,35.817961
std,103.923736,86.313283,64.16564,21.642256,5.968103,78.112692,79.723362,63.983704,16.705742
min,102.000000,0.000000,0.00000,121.800000,0.000000,801.000000,594.000000,1.000000,2.330000
25%,190.700000,0.000000,0.00000,164.900000,0.000000,932.000000,733.250000,7.000000,23.710000
50%,272.700000,20.000000,0.00000,185.000000,6.400000,968.000000,778.500000,28.000000,34.445000
75%,350.000000,144.150000,118.30000,192.900000,10.200000,1029.400000,823.550000,56.000000,46.135000
max,540.000000,359.400000,200.10000,247.000000,32.200000,1145.000000,992.600000,365.000000,82.600000


1. **Все значения неотрицательны**, что полностью соответствует физическому смыслу признаков (масса и время).

2. **Разность масштабов:** Разброс значений колоссален (от единиц у `Superplasticizer` до тысяч у `Coarse Aggregate`). Это означает обязательную необходимость скалирования признаков на следующих этапах.

3. **Естественные нули:** Минимум (а иногда и медиана) у шлака, золы и пластификатора равен 0. Это нормально — данные добавки применяются не в каждой смеси.

4. **Специфика `Age`:** Медиана составляет 28 дней (стандарт тестирования), но есть сильный правый хвост (максимум 365). Такой разброс подтверждает нашу гипотезу о необходимости логарифмирования возраста для обучения. Разница между 1 и 28 днем огромная (там достигается 90% прочности), а дальше уже почти нулевая. Но машина этого не понимает и подумает, что за такое время прочность должна была вырасти в разы. Как спасает логарифм? В математике логарифмирование — это функция, которая "сжимает" большие числа и "растягивает" маленькие. Если мы вместо самих дней возьмем их логарифм, то машина увидит, что промежуток от 1 до 28 дней имеет большой вес, а огромный скачок от 180 до 365 дней превратится в маленький шажок.

In [9]:
df.nunique()

Cement kg/m^3                    273
Blast Furnace Slag kg/m^3        180
Fly Ash kg/m^3                   146
Water kg/m^3                     188
Superplasticizer kg/m^3          110
Coarse Aggregate kg/m^3          274
Fine Aggregate kg/m^3            280
Age day                           14
Concrete compressive strength    845
dtype: int64

1. В целевой переменной (`Concrete compressive strength`) целых 845 разных значений на 1030 строк. То есть мы предсказываем конкретное дробное число, а не просто категории "крепкий/хрупкий". Значит, это точно задача регрессии.

2. Во всех признаках состава смеси уникальных значений больше сотни. Категориальных колонок (текстовых или где всего 2-3 варианта ответа) у нас вообще нет. Получается шаг с кодированием категорий можно будет просто пропустить.

3. `Age` имеет всего 14 уникальных значений. Это значит, что бетон тестировали не каждый день подряд, а только в строго определенные контрольные дни (например, на 7-й, 14-й, 28-й день и т.д.), как это обычно и делают в лабораториях.

## 1.3. Разделение выборки на обучающую и тестовую выборки

In [10]:
y = df['Concrete compressive strength']
X = df.drop(['Concrete compressive strength'], axis = 1)

In [11]:
y

0       79.99
1       61.89
2       40.27
3       41.05
4       44.30
        ...  
1025    44.28
1026    31.18
1027    23.70
1028    32.77
1029    32.40
Name: Concrete compressive strength, Length: 1030, dtype: float64

In [12]:
X

,Cement kg/m^3,Blast Furnace Slag kg/m^3,Fly Ash kg/m^3,Water kg/m^3,Superplasticizer kg/m^3,Coarse Aggregate kg/m^3,Fine Aggregate kg/m^3,Age day
0,540.0,0.0,0.0,NaN,2.5,1040.0,676.0,28.0
1,540.0,0.0,0.0,162.0,2.5,1055.0,676.0,28.0
2,332.5,142.5,0.0,228.0,0.0,932.0,594.0,270.0
3,332.5,142.5,0.0,228.0,0.0,932.0,594.0,365.0
4,198.6,132.4,0.0,192.0,0.0,978.4,825.5,360.0
...,...,...,...,...,...,...,...,...
1025,276.4,NaN,90.3,179.6,8.9,NaN,768.3,28.0
1026,322.2,0.0,115.6,NaN,NaN,817.9,813.4,28.0
1027,148.5,139.4,108.6,192.7,6.1,892.4,780.0,28.0
1028,159.1,186.7,NaN,175.6,11.3,989.6,788.9,28.0


In [14]:
X_train, X_test, y_train, y_test = train_test_split(X, y, shuffle=True, random_state=42, test_size=0.2)

In [15]:
print(f"Размер X_train: {X_train.shape}, y_train: {y_train.shape}")
print(f"Размер X_test:  {X_test.shape}, y_test:  {y_test.shape}")

Размер X_train: (824, 8), y_train: (824,)
Размер X_test:  (206, 8), y_test:  (206,)


## 1.4. Обработка вещественных признаков (заполнение пропусков)

## 1.5. Кодирование категориальных признаков

## 1.6. Детекция выбросов и аномалий в данных

## 1.7. Подведение итогов раздела 1
<!-- А таких разделах обычно подытоживается вся информация о тех действиях, которые вы делали в разделе -->

# 2. Генерация новых признаков

## 2.1. Корреляционный анализ входных признаков (построение тепловых карт корреляции)

## 2.2. Исправление проблемы мультиколлинеарности в данных

## 2.3. Скалирование данных

## 2.4. Подведение итогов раздела 2
<!-- А таких разделах обычно подытоживается вся информация о тех действиях, которые вы делали в разделе -->

# 3. Выбор моделей ML и метрик

`ячейки для выбора моделей ML`
<!-- Третий раздел самый богатый на писанину. Вам необходимо сначала обосновать выбор моделей, а затем все отобранные модели описать (расписать концептуально, как они работают) -->

`ячейки для выбора метрик`
<!-- И снова писанина: необходимо не только описать, какие метрики вы используете, но привести формулы их вычисления -->

In [16]:
# ячейка для импорта моделей из пула

In [17]:
# ячейки для формирования сеток гиперпараметров

In [18]:
# ячейка для импорта метрик из пула

# 4. Обучение моделей ML и подбор гиперпараметров

In [19]:
# ячейки для подбора гиперпараметров и обучения лучших моделей

# 5. Вычисление метрик на новых данных

In [20]:
# ячейки для вычисления метрик на лучших моделях и их сопоставление

# 6. Результат работы

`ячейка для подведения итогов по всей работе`
<!-- Тут обычно пишется, какая модель показала себя лучше, с какими метриками и рекомендуется ли ее выводить в прод -->